# 📄 PDF Document Loader and Chunking

This notebook demonstrates the **Data Ingestion** phase of the RAG pipeline. It handles:
1. Scanning and recursively finding PDF files in a directory.
2. Loading the PDFs page-by-page using `PyPDFLoader`.
3. Enriched metadata tagging (`source_file`, `file_type`).
4. Splitting the loaded text into semantically cohesive chunks using `RecursiveCharacterTextSplitter`.

In [ ]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("✅ Imports loaded successfully.")

In [ ]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory recursively and extract pages."""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add custom metadata fields to each page
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error loading {pdf_file.name}: {e}")
            
    print(f"\nTotal documents/pages loaded: {len(all_documents)}")
    return all_documents

In [ ]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split loaded document pages into smaller chunks for optimal vector search."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} pages into {len(split_docs)} chunks")
    
    # Show an example chunk if any are generated
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:250]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [ ]:
# Define the data directory
data_directory = "../data"

# Ensure the data directory exists
os.makedirs(data_directory, exist_ok=True)

print("Step 1: Scanning and loading PDF documents...")
loaded_docs = process_all_pdfs(data_directory)

if loaded_docs:
    print("\nStep 2: Splitting documents into text chunks...")
    chunks = split_documents(loaded_docs, chunk_size=1000, chunk_overlap=200)
else:
    print("\nNo PDF documents found. Please place PDF files in the 'data' directory at the root.")